# Exercise 7 — Faults vs Disturbances

Not all abnormal responses mean the same thing. A sluggish response could be
increased load, worn bearings, sensor drift, or a detuned controller. This
exercise builds intuition for **diagnosing root causes from response behavior**.

We will run the *same* PID controller on the *same* plant under six different
fault and disturbance scenarios, then compare the resulting responses
side-by-side so you can learn to read the signatures.

## Baseline Plant & Controller

We use a second-order plant (like a motor driving a conveyor) with PID control.
All scenarios start from the same baseline so the *only* difference is the
injected fault or disturbance.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def simulate_scenario(
    # Plant
    wn=2.0, zeta=0.4, K_plant=1.0, J=0.5,
    # PID
    Kp=5.0, Ki=2.0, Kd=0.5,
    u_min=0.0, u_max=10.0,
    # Modifications
    sensor_bias=0.0,        # added to measurement [m/s]
    actuator_scale=1.0,     # multiplied onto control output (1.0 = healthy)
    time_delay_steps=0,     # pure delay in control action
    extra_friction=0.0,     # extra damping force [N]
    dist_load=0.0,          # constant disturbance load [N]
    # Sim
    dt=0.005, t_end=30.0,
    target=1.0,
):
    t = np.arange(0, t_end, dt)
    n = len(t)
    ref = np.full(n, target)
    # Add step disturbance at t=10s
    dist = np.zeros(n)
    dist[t >= 10.0] = dist_load

    y = np.zeros(n)
    dy = np.zeros(n)
    u = np.zeros(n)

    integral_e = 0.0
    prev_e = 0.0
    u_buffer = np.zeros(max(time_delay_steps + 1, 1))

    for i in range(1, n):
        # Sensor reads plant output + bias
        y_measured = y[i - 1] + sensor_bias
        e = ref[i - 1] - y_measured
        integral_e += e * dt
        de = (e - prev_e) / dt
        prev_e = e

        u_raw = Kp * e + Ki * integral_e + Kd * de
        u_cmd = np.clip(u_raw, u_min, u_max)
        if u_cmd != u_raw:
            integral_e -= e * dt  # anti-windup

        # Delay and actuator weakness
        if time_delay_steps > 0:
            u_buffer = np.roll(u_buffer, 1)
            u_buffer[0] = u_cmd * actuator_scale
            u_applied = u_buffer[-1]
        else:
            u_applied = u_cmd * actuator_scale
        u[i] = u_applied

        # Plant: y'' + 2*zeta*wn*y' + wn^2*y = wn^2*K*u - dist/J - extra_friction*y'/J
        ddy = (
            wn**2 * K_plant * u_applied
            - 2 * zeta * wn * dy[i - 1]
            - wn**2 * y[i - 1]
            - dist[i - 1] / J
            - extra_friction * dy[i - 1] / J
        )
        dy[i] = dy[i - 1] + ddy * dt
        y[i] = max(y[i - 1] + dy[i] * dt, 0.0)

    return dict(t=t, y=y, ref=ref, u=u, dist=dist)

## Scenario Definitions

We run **6 scenarios** and compare them all. Each scenario changes exactly one
thing relative to the nominal baseline.

In [ ]:
scenarios = {
    "Nominal": dict(),
    "Heavy load": dict(dist_load=3.0),
    "Increased friction": dict(extra_friction=2.0),
    "Sensor bias (+0.2)": dict(sensor_bias=0.2),
    "Weak actuator (60%)": dict(actuator_scale=0.6),
    "Time delay (100ms)": dict(time_delay_steps=20),
}

results = {}
for name, overrides in scenarios.items():
    results[name] = simulate_scenario(**overrides)

## Side-by-Side Comparison

In [ ]:
colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A", "#FF6692"]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Plant Output (Speed)", "Control Effort"),
    vertical_spacing=0.10,
)

for (name, res), color in zip(results.items(), colors):
    fig.add_trace(
        go.Scatter(
            x=res["t"], y=res["y"], mode="lines",
            name=name, line=dict(color=color, width=2),
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=res["t"], y=res["u"], mode="lines",
            name=name, line=dict(color=color, width=1.5),
            showlegend=False,
        ),
        row=2, col=1,
    )

# Add target reference
fig.add_trace(
    go.Scatter(
        x=results["Nominal"]["t"], y=results["Nominal"]["ref"],
        mode="lines", name="Target",
        line=dict(color="black", dash="dash", width=1.5),
    ),
    row=1, col=1,
)

fig.update_yaxes(title_text="Speed [m/s]", row=1, col=1)
fig.update_yaxes(title_text="Voltage [V]", row=2, col=1)
fig.update_xaxes(title_text="Time [s]", row=2, col=1)
fig.update_layout(
    template="plotly_white", height=600,
    title_text="Fault vs Disturbance Comparison",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(t=80, b=40),
)
fig.show()

## Scenario-by-Scenario Analysis

### Nominal

Baseline: well-tuned controller tracks the setpoint with modest overshoot and
quick settling. This is our reference for comparison.

### Heavy Load (Disturbance)

**What changed:** External load force applied at t=10s.

**Signature:** Output dips after load onset, then recovers. Control effort
increases to compensate.

**Diagnosis:** Disturbance — not a fault. The controller handles it.

**Action:** Continue running. If load is persistent, consider feedforward
compensation.

### Increased Friction (Degradation)

**What changed:** Extra friction in bearings/belt — plant dynamics are slower.

**Signature:** Sluggish response, slower settling, slightly higher steady control
effort.

**Diagnosis:** Fault / degradation — the plant itself has changed.

**Action:** Inspect bearings, lubrication, belt tension. Schedule maintenance if
trending.

### Sensor Bias (Sensing Problem)

**What changed:** Sensor reads 0.2 m/s higher than actual speed.

**Signature:** Steady-state offset — output settles BELOW target because the
controller "thinks" it is on target.

**Diagnosis:** Sensor calibration issue.

**Action:** Recalibrate sensor. A persistent offset that appears suddenly is a
red flag.

### Weak Actuator (Fault)

**What changed:** Actuator only delivers 60% of commanded force (degraded motor,
slipping belt).

**Signature:** Slower rise, integral action compensates over time but control
effort is higher. May hit saturation sooner.

**Diagnosis:** Actuator degradation — the controller can partially compensate,
masking the problem.

**Action:** This is the dangerous one. The system "looks OK" until the actuator
degrades further and can no longer compensate. Inspect actuator, check drive
current vs command.

### Time Delay (Control Problem)

**What changed:** 100ms of pure delay added to the control loop.

**Signature:** Increased oscillation and overshoot. The controller is reacting to
stale information.

**Diagnosis:** Communication lag, slow sensor, or processing delay.

**Action:** Retune controller (reduce gains to accommodate delay), or investigate
and reduce the source of delay.

## Summary: Diagnosis Decision Table

| Scenario | Response Signature | Root Cause Category | Recommended Action |
|---|---|---|---|
| Nominal | Clean tracking | — | Continue running |
| Heavy load | Dip + recovery | Disturbance | Monitor / feedforward |
| Increased friction | Sluggish | Degradation / fault | Inspect + maintenance |
| Sensor bias | Steady-state offset | Sensing problem | Recalibrate sensor |
| Weak actuator | Slow rise, high effort | Actuator fault | Inspect actuator |
| Time delay | Oscillation / overshoot | Control / comm issue | Retune or fix delay |

**Key takeaway:** The same controller on the same plant can produce very
different-looking responses depending on what went wrong. Learning to read these
signatures is the first step toward predictive maintenance.

### Student Challenge

Run `simulate_scenario` with **two faults combined** (e.g., sensor bias AND weak
actuator). Describe:

1. What the response looks like.
2. Which fault dominates the behavior.
3. What action you would recommend.

In [ ]:
# ---- Student challenge: combine two faults ----
# Example: result = simulate_scenario(sensor_bias=0.15, actuator_scale=0.7)